# OSS Artifact Cluster Assignment

This notebook assigns open-source repository artifacts to clusters discovered in the main analysis.

**Workflow:**
1. Load the trained cluster model (UMAP + HDBSCAN)
2. Load OSS embeddings and text content
3. **Filter OSS data** (remove README/LICENSE, keep AI-tool whitelist)
4. Transform OSS embeddings to cluster space
5. Assign artifacts to clusters
6. Analyze cluster semantics using text content

## Setup

In [ ]:
import os

# ============================================================
# LLM CONFIGURATION
# ============================================================
# ANTHROPIC_API_KEY is loaded from .env via src/__init__.py
# Copy .env.example to .env and set your key there

# Model selection for cluster analysis
ANTHROPIC_MODEL = "claude-sonnet-4-20250514"  # Recommended for analysis
# Alternatives:
#   "claude-3-5-haiku-20241022"  - Faster, cheaper, less nuanced
#   "claude-opus-4-5-20251101"   - Most capable, higher cost

# Add project root to path so src/__init__.py loads .env
import sys
sys.path.insert(0, os.path.abspath('..'))
import src  # triggers .env loading

# Detect available APIs
USE_ANTHROPIC = "ANTHROPIC_API_KEY" in os.environ
USE_OPENAI = "OPENAI_API_KEY" in os.environ

print(f"LLM Model: {ANTHROPIC_MODEL}")
print(f"Anthropic API: {'\u2713' if USE_ANTHROPIC else '\u2717'}")

# ============================================================
# IMPORTS
# ============================================================
import pickle
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter

# Visualization
import plotly.express as px
import plotly.graph_objects as go

# UMAP for visualization
import umap

# HDBSCAN for approximate_predict
try:
    import hdbscan
    HAS_HDBSCAN = True
except ImportError:
    print("Installing hdbscan...")
    import subprocess
    subprocess.check_call(['pip', 'install', 'hdbscan', '-q'])
    import hdbscan
    HAS_HDBSCAN = True

# ============================================================
# CONFIGURATION: Select which cluster model to use
# ============================================================
CLUSTER_MODEL_DIR = Path('../data/cluster_model')

# List available models
print("\nAvailable cluster models:")
print("-" * 60)
available_models = sorted(CLUSTER_MODEL_DIR.glob('cluster_model*.pkl'))
for i, model_path in enumerate(available_models):
    size_kb = model_path.stat().st_size / 1024
    name = model_path.name
    # Mark latest
    marker = " <- LATEST" if name == "cluster_model.pkl" else ""
    print(f"  [{i}] {name} ({size_kb:.1f} KB){marker}")

print("-" * 60)

# SELECT YOUR MODEL HERE (by filename or index)
CLUSTER_MODEL_NAME = "cluster_model.pkl"  # Use latest

# ============================================================

CLUSTER_MODEL_PATH = CLUSTER_MODEL_DIR / CLUSTER_MODEL_NAME
DATA_DIR = Path('../output_oss')

print(f"\n\u2713 Selected cluster model: {CLUSTER_MODEL_NAME}")

## 1. Load Cluster Model

In [ ]:
# Load the trained cluster model
with open(CLUSTER_MODEL_PATH, 'rb') as f:
    cluster_model = pickle.load(f)

print("=" * 70)
print("CLUSTER MODEL LOADED")
print("=" * 70)
print(f"\nModel created: {cluster_model['created_at']}")
print(f"Training samples: {cluster_model['n_training_samples']:,}")
print(f"Number of clusters: {cluster_model['n_clusters']}")
print(f"Embedding dimension: {cluster_model['embedding_dim']}")

print(f"\nParameters:")
for k, v in cluster_model['parameters'].items():
    print(f"  {k}: {v}")

# Extract components
umap_reducer = cluster_model['umap_reducer']
hdbscan_clusterer = cluster_model['hdbscan_clusterer']
cluster_centroids = cluster_model['cluster_centroids']
cluster_metadata = cluster_model['cluster_metadata']

# Generate prediction data for approximate_predict (if not already present)
if not hasattr(hdbscan_clusterer, '_prediction_data') or hdbscan_clusterer._prediction_data is None:
    print("\nGenerating prediction data for HDBSCAN...")
    hdbscan_clusterer.generate_prediction_data()
    print("✓ Prediction data generated")

print(f"\nCluster sizes:")
for c in sorted(cluster_metadata.keys()):
    print(f"  Cluster {c}: {cluster_metadata[c]['size']:,} files")

## 2. Load OSS Data

In [ ]:
# Load OSS embeddings
with open(DATA_DIR / 'oss_embeddings.pkl', 'rb') as f:
    oss_embeddings_data = pickle.load(f)

oss_embeddings = oss_embeddings_data['embeddings']
oss_file_ids = oss_embeddings_data['file_ids']

print(f"OSS embeddings shape: {oss_embeddings.shape}")
print(f"Number of files: {len(oss_file_ids)}")

# Load OSS text content
with open(DATA_DIR / 'oss_text_content.pkl', 'rb') as f:
    oss_text_data = pickle.load(f)

# Build dict mapping file_id -> text
text_file_ids = oss_text_data.get('file_ids', [])
text_contents = oss_text_data.get('texts', [])
oss_texts = dict(zip(text_file_ids, text_contents))

print(f"Text content available: {len(oss_texts)} files")

# Load metadata
oss_metadata = pd.read_csv(DATA_DIR / 'oss_artifacts.csv')
print(f"Metadata rows: {len(oss_metadata)}")
print(f"Columns: {oss_metadata.columns.tolist()}")

## 2b. Filter OSS Data

Apply the same filters as Notebook 2 (embedding_filtration) to ensure consistency.
This removes README files and other non-AI-specific artifacts.

In [ ]:
# ============================================================
# FILTER: Remove README files
# ============================================================

print("=" * 70)
print("FILTERING OSS DATA")
print("=" * 70)

n_before = len(oss_file_ids)
print(f"\nBefore filtering: {n_before:,} files")

# Get artifact names for filtering
file_id_to_artifact = dict(zip(oss_metadata['file_id'], oss_metadata['artifact_name']))

# Filter out README files only
keep_mask = []
readme_count = 0

for file_id in oss_file_ids:
    artifact_name = file_id_to_artifact.get(file_id, '')
    artifact_lower = artifact_name.lower() if artifact_name else ''
    
    if artifact_lower in ['readme.md', 'readme', 'readme.txt']:
        keep_mask.append(False)
        readme_count += 1
    else:
        keep_mask.append(True)

keep_mask = np.array(keep_mask)

# Apply filter
oss_embeddings = oss_embeddings[keep_mask]
oss_file_ids = [fid for fid, keep in zip(oss_file_ids, keep_mask) if keep]
oss_metadata = oss_metadata[oss_metadata['file_id'].isin(oss_file_ids)].copy()
oss_texts = {fid: txt for fid, txt in oss_texts.items() if fid in set(oss_file_ids)}

n_after = len(oss_file_ids)

print(f"\nFiltered out: {readme_count:,} README files")
print(f"After filtering: {n_after:,} files")
print(f"\n✓ Filtering complete")

## 3. Transform OSS Embeddings

Apply the trained UMAP reducer to project OSS embeddings to the same space.

In [ ]:
# Transform OSS embeddings using the trained UMAP
print(f"Transforming {len(oss_embeddings):,} embeddings...")
print(f"  From: {oss_embeddings.shape[1]}D")
print(f"  To: {cluster_model['parameters']['umap_n_components']}D")

oss_umap = umap_reducer.transform(oss_embeddings)
print(f"\n✓ Transformed shape: {oss_umap.shape}")

## 4. Assign to Clusters

Two methods available:
1. **HDBSCAN approximate_predict**: Uses the trained model's structure
2. **Nearest centroid**: Simple distance-based assignment

In [ ]:
# Method 1: HDBSCAN approximate_predict
# This respects the cluster structure but may assign many points to noise (-1)
print("Method 1: HDBSCAN approximate_predict")
print("-" * 50)

oss_labels_hdbscan, oss_strengths = hdbscan.approximate_predict(
    hdbscan_clusterer, 
    oss_umap
)

n_assigned = (oss_labels_hdbscan != -1).sum()
n_noise = (oss_labels_hdbscan == -1).sum()
print(f"  Assigned to clusters: {n_assigned:,} ({n_assigned/len(oss_labels_hdbscan)*100:.1f}%)")
print(f"  Noise (unassigned): {n_noise:,} ({n_noise/len(oss_labels_hdbscan)*100:.1f}%)")

print(f"\n  Cluster distribution:")
for c in sorted(set(oss_labels_hdbscan)):
    count = (oss_labels_hdbscan == c).sum()
    label = 'Noise' if c == -1 else f'Cluster {c}'
    print(f"    {label}: {count:,}")

In [ ]:
# Method 2: Nearest centroid assignment
# Every point gets assigned to the closest cluster
print("Method 2: Nearest Centroid Assignment")
print("-" * 50)

from scipy.spatial.distance import cdist

# Build centroid matrix
centroid_ids = sorted(cluster_centroids.keys())
centroid_matrix = np.array([cluster_centroids[c] for c in centroid_ids])

# Compute distances
distances = cdist(oss_umap, centroid_matrix, metric='euclidean')

# Assign to nearest
nearest_idx = distances.argmin(axis=1)
oss_labels_centroid = np.array([centroid_ids[i] for i in nearest_idx])
oss_distances = distances.min(axis=1)

print(f"  All {len(oss_labels_centroid):,} points assigned")
print(f"  Mean distance to centroid: {oss_distances.mean():.4f}")
print(f"  Max distance: {oss_distances.max():.4f}")

print(f"\n  Cluster distribution:")
for c in sorted(set(oss_labels_centroid)):
    count = (oss_labels_centroid == c).sum()
    print(f"    Cluster {c}: {count:,}")

In [ ]:
# Choose which method to use for analysis
# Options: 'hdbscan' or 'centroid'
ASSIGNMENT_METHOD = 'hdbscan'  # Using approximate_predict for high-confidence semantic analysis

if ASSIGNMENT_METHOD == 'hdbscan':
    oss_labels = oss_labels_hdbscan
    print("Using HDBSCAN approximate_predict assignments")
else:
    oss_labels = oss_labels_centroid
    print("Using nearest centroid assignments")

# Build results DataFrame
oss_results = pd.DataFrame({
    'file_id': oss_file_ids,
    'cluster': oss_labels,
    'cluster_hdbscan': oss_labels_hdbscan,
    'cluster_centroid': oss_labels_centroid,
    'distance_to_centroid': oss_distances,
    'hdbscan_strength': oss_strengths,
})

# Merge with metadata (deduplicate to avoid row explosion)
metadata_dedup = oss_metadata[['file_id', 'repo_name', 'artifact_name', 'tool_name']].drop_duplicates(subset='file_id')
print(f"Metadata: {len(oss_metadata)} rows -> {len(metadata_dedup)} unique file_ids")

oss_results = oss_results.merge(
    metadata_dedup, 
    on='file_id', 
    how='left'
)

print(f"Results DataFrame: {len(oss_results)} rows")
oss_results.head(10)

## 5. Visualize Assignments

Two visualizations:
1. **HDBSCAN approximate_predict** (primary) - High-confidence assignments with noise shown in gray
2. **Method comparison** - Side-by-side comparison of HDBSCAN vs Nearest Centroid

**Why HDBSCAN for semantic analysis:**
- Only assigns points that confidently fit the learned cluster density structure
- 73% "noise" = artifacts that don't match enterprise patterns (itself a finding)
- Remaining 27% are high-confidence matches suitable for semantic interpretation

In [ ]:
# Reduce to 2D for visualization
print("Reducing to 2D for visualization...")
reducer_2d = umap.UMAP(
    n_neighbors=15,
    n_components=2,
    min_dist=0.1,
    metric='euclidean',
    random_state=42,
    n_jobs=1
)
oss_coords_2d = reducer_2d.fit_transform(oss_umap)

oss_results['umap_x'] = oss_coords_2d[:, 0]
oss_results['umap_y'] = oss_coords_2d[:, 1]

print(f"✓ 2D coordinates computed")

In [ ]:
# Scatter plot of cluster assignments - HDBSCAN approximate_predict
# This visualization uses the HDBSCAN method (high-confidence assignments)

# Create labels for HDBSCAN method
oss_results['hdbscan_label'] = oss_results['cluster_hdbscan'].apply(
    lambda x: 'Noise (unassigned)' if x == -1 else f'Cluster {x}'
)

# Calculate statistics
n_clustered = (oss_results['cluster_hdbscan'] != -1).sum()
n_noise = (oss_results['cluster_hdbscan'] == -1).sum()
pct_clustered = n_clustered / len(oss_results) * 100

# Sort order: clusters first, noise last
cluster_order = [f'Cluster {i}' for i in sorted(set(oss_results['cluster_hdbscan']) - {-1})] + ['Noise (unassigned)']

fig_hdbscan = px.scatter(
    oss_results,
    x='umap_x',
    y='umap_y',
    color='hdbscan_label',
    category_orders={'hdbscan_label': cluster_order},
    hover_data=['artifact_name', 'repo_name', 'tool_name', 'hdbscan_strength'],
    title=f'OSS Artifacts: HDBSCAN approximate_predict<br><sup>Clustered: {n_clustered:,} ({pct_clustered:.1f}%) | Noise: {n_noise:,} ({100-pct_clustered:.1f}%)</sup>',
    width=1000,
    height=700,
)

# Style: make noise points smaller and gray, clustered points larger
fig_hdbscan.for_each_trace(
    lambda t: t.update(marker=dict(size=3, opacity=0.3, color='lightgray')) 
    if 'Noise' in t.name 
    else t.update(marker=dict(size=6, opacity=0.8))
)

fig_hdbscan.update_layout(
    xaxis_title='UMAP Dimension 1',
    yaxis_title='UMAP Dimension 2',
    legend_title='Cluster Assignment',
)
fig_hdbscan.show()

# Print cluster distribution
print(f"\nHDBSCAN approximate_predict Cluster Distribution:")
print(f"{'='*50}")
for cluster_id in sorted(set(oss_results['cluster_hdbscan'])):
    count = (oss_results['cluster_hdbscan'] == cluster_id).sum()
    label = 'Noise' if cluster_id == -1 else f'Cluster {cluster_id}'
    pct = count / len(oss_results) * 100
    bar = '█' * int(pct / 2)
    print(f"  {label:20s}: {count:5,} ({pct:5.1f}%) {bar}")

In [ ]:
# Side-by-side comparison: HDBSCAN approximate_predict vs Nearest Centroid
from plotly.subplots import make_subplots

# Create labels for centroid method
oss_results['centroid_label'] = oss_results['cluster_centroid'].apply(lambda x: f'Cluster {x}')

fig_compare = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        f'HDBSCAN approximate_predict<br><sup>Coverage: {pct_clustered:.1f}% (high confidence)</sup>',
        f'Nearest Centroid<br><sup>Coverage: 100% (all assigned)</sup>'
    ),
    horizontal_spacing=0.08
)

# Left plot: HDBSCAN (clustered points only, noise in gray)
for cluster_id in sorted(set(oss_results['cluster_hdbscan'])):
    mask = oss_results['cluster_hdbscan'] == cluster_id
    if cluster_id == -1:
        fig_compare.add_trace(
            go.Scatter(
                x=oss_results.loc[mask, 'umap_x'],
                y=oss_results.loc[mask, 'umap_y'],
                mode='markers',
                marker=dict(size=2, color='lightgray', opacity=0.2),
                name='Noise',
                showlegend=False
            ),
            row=1, col=1
        )
    else:
        fig_compare.add_trace(
            go.Scatter(
                x=oss_results.loc[mask, 'umap_x'],
                y=oss_results.loc[mask, 'umap_y'],
                mode='markers',
                marker=dict(size=5, opacity=0.7),
                name=f'C{cluster_id}',
                legendgroup=f'cluster_{cluster_id}'
            ),
            row=1, col=1
        )

# Right plot: Nearest Centroid (all points colored)
for cluster_id in sorted(set(oss_results['cluster_centroid'])):
    mask = oss_results['cluster_centroid'] == cluster_id
    fig_compare.add_trace(
        go.Scatter(
            x=oss_results.loc[mask, 'umap_x'],
            y=oss_results.loc[mask, 'umap_y'],
            mode='markers',
            marker=dict(size=4, opacity=0.6),
            name=f'C{cluster_id}',
            legendgroup=f'cluster_{cluster_id}',
            showlegend=False
        ),
        row=1, col=2
    )

fig_compare.update_layout(
    title='Method Comparison: OSS Artifact Cluster Assignment',
    height=550,
    width=1200,
    legend=dict(orientation='h', yanchor='bottom', y=-0.15, xanchor='center', x=0.5)
)

fig_compare.update_xaxes(title_text='UMAP Dim 1', row=1, col=1)
fig_compare.update_xaxes(title_text='UMAP Dim 1', row=1, col=2)
fig_compare.update_yaxes(title_text='UMAP Dim 2', row=1, col=1)

fig_compare.show()

# Summary comparison table
print("\n" + "=" * 70)
print("METHOD COMPARISON SUMMARY")
print("=" * 70)
print(f"\n{'Metric':<35} {'HDBSCAN':>15} {'Centroid':>15}")
print("-" * 70)
print(f"{'Artifacts assigned to clusters':<35} {n_clustered:>15,} {len(oss_results):>15,}")
print(f"{'Coverage %':<35} {pct_clustered:>14.1f}% {100.0:>14.1f}%")
print(f"{'Noise/unassigned':<35} {n_noise:>15,} {0:>15,}")
print(f"{'Confidence level':<35} {'High':>15} {'Variable':>15}")
print("-" * 70)
print("\nRecommendation: HDBSCAN for semantic analysis (removes low-confidence matches)")

## 6. Exemplar-Based Cluster Semantic Analysis

Select representative files from each cluster and analyze with LLM using full content.

**Method:**
- Select 7-10 files nearest to cluster centroid (most typical)
- Add 1-2 boundary files (furthest from centroid) for diversity
- Ensure repository diversity - sample from different repos when possible
- Feed FULL file content to LLM (no truncation)
- Ask research-focused questions about AI coding tool usage patterns

In [ ]:
def select_cluster_exemplars(cluster_id, n_centroid=8, n_boundary=2):
    """
    Select exemplar files: nearest to centroid + boundary cases.
    Ensures repository diversity - picks from different repos when possible.
    
    Args:
        cluster_id: The cluster to analyze
        n_centroid: Number of centroid-nearest files to select (default 8)
        n_boundary: Number of boundary files to select (default 2)
    
    Returns:
        DataFrame with columns: file_id, artifact_name, repo_name, distance_to_centroid, 
                                text, selection_type, file_size
    """
    from scipy.spatial.distance import cdist
    
    # Get files in this cluster
    cluster_mask = oss_results['cluster'] == cluster_id
    cluster_files = oss_results[cluster_mask].copy()
    
    if len(cluster_files) == 0:
        print(f"Warning: Cluster {cluster_id} has no files")
        return pd.DataFrame()
    
    # Get UMAP coordinates for files in this cluster
    cluster_indices = oss_results[cluster_mask].index.tolist()
    cluster_umap_coords = oss_umap[cluster_indices]
    
    # Get cluster centroid
    centroid = cluster_centroids[cluster_id]
    
    # Compute distances to centroid
    distances = cdist(cluster_umap_coords, [centroid], metric='euclidean').flatten()
    cluster_files['distance_to_centroid'] = distances
    
    # Add text content
    cluster_files['text'] = cluster_files['file_id'].map(oss_texts)
    cluster_files['text_available'] = cluster_files['text'].notna() & (cluster_files['text'] != '')
    
    # Filter to files with text content
    files_with_text = cluster_files[cluster_files['text_available']].copy()
    
    if len(files_with_text) == 0:
        print(f"Warning: Cluster {cluster_id} has no files with text content")
        return pd.DataFrame()
    
    # Group by repository and get closest file per repo
    repo_groups = files_with_text.groupby('repo_name')
    best_per_repo = []
    
    for repo_name, group in repo_groups:
        # Get file closest to centroid from this repo
        closest_idx = group['distance_to_centroid'].idxmin()
        closest_file = group.loc[closest_idx].copy()
        closest_file['repo_min_distance'] = group['distance_to_centroid'].min()
        best_per_repo.append(closest_file)
    
    best_per_repo_df = pd.DataFrame(best_per_repo)
    
    # Rank repos by their best file's distance to centroid
    best_per_repo_df = best_per_repo_df.sort_values('distance_to_centroid')
    
    # Select top n_centroid files (ensuring repo diversity)
    n_repos_available = len(best_per_repo_df)
    n_to_select = min(n_centroid, n_repos_available)
    
    centroid_exemplars = best_per_repo_df.head(n_to_select).copy()
    centroid_exemplars['selection_type'] = 'centroid_nearest'
    
    # For boundary files: select from repos NOT already selected
    selected_repos = set(centroid_exemplars['repo_name'])
    boundary_candidates = files_with_text[~files_with_text['repo_name'].isin(selected_repos)]
    
    if len(boundary_candidates) < n_boundary:
        # If not enough repos, select from already-used repos (furthest files)
        boundary_candidates = files_with_text[~files_with_text['file_id'].isin(centroid_exemplars['file_id'])]
    
    # Sort by distance descending (furthest from centroid)
    boundary_candidates = boundary_candidates.sort_values('distance_to_centroid', ascending=False)
    
    # Select top n_boundary boundary files (prefer different repos)
    boundary_exemplars = []
    boundary_repos_used = set()
    
    for _, row in boundary_candidates.iterrows():
        if len(boundary_exemplars) >= n_boundary:
            break
        # Prefer different repos
        if row['repo_name'] not in boundary_repos_used or len(boundary_exemplars) < n_boundary:
            boundary_exemplars.append(row)
            boundary_repos_used.add(row['repo_name'])
    
    boundary_exemplars_df = pd.DataFrame(boundary_exemplars)
    if not boundary_exemplars_df.empty:
        boundary_exemplars_df['selection_type'] = 'boundary'
    
    # Combine centroid and boundary exemplars
    all_exemplars = pd.concat([centroid_exemplars, boundary_exemplars_df], ignore_index=True)
    
    # Get file sizes from text
    all_exemplars['file_size'] = all_exemplars['text'].apply(lambda x: len(x) if x else 0)
    
    # Select and order columns
    columns = ['file_id', 'artifact_name', 'repo_name', 'distance_to_centroid', 
               'file_size', 'selection_type', 'text']
    
    result = all_exemplars[columns].copy()
    
    return result


print("✓ Exemplar selection function defined")
print(f"  - Selects files nearest to cluster centroid")
print(f"  - Ensures repository diversity")
print(f"  - Adds boundary files for coverage")

In [ ]:
# Initialize LLM client (uses configuration from Setup cell)
if USE_ANTHROPIC:
    import anthropic
    client = anthropic.Anthropic()
    print(f"✓ Using Anthropic API with model: {ANTHROPIC_MODEL}")
elif USE_OPENAI:
    from openai import OpenAI
    client = OpenAI()
    print("✓ Using OpenAI API")
else:
    raise ValueError("No API key found! Check Setup cell configuration.")


def analyze_cluster_exemplars(cluster_id, exemplars, max_file_size=50000):
    """
    Analyze cluster using FULL exemplar file content.
    
    Args:
        cluster_id: The cluster being analyzed
        exemplars: DataFrame from select_cluster_exemplars()
        max_file_size: Max chars per file (warn if exceeded, default 50KB)
    
    Returns:
        dict with 'summary' and 'file_warnings'
    """
    if exemplars.empty:
        return {'summary': 'No exemplars available for analysis.', 'file_warnings': []}
    
    file_warnings = []
    file_contents = []
    
    for _, row in exemplars.iterrows():
        file_id = row['file_id']
        text = row['text']
        selection_type = row['selection_type']
        
        if not text or len(text) == 0:
            file_warnings.append(f"{file_id}: No text content")
            continue
        
        if len(text) > max_file_size:
            file_warnings.append(f"{file_id}: Large file ({len(text):,} chars), may affect analysis")
            # Still include full content - let LLM handle it
        
        file_contents.append(f"""
=== FILE: {file_id} ({selection_type}) ===
Repository: {row['repo_name']}
Artifact: {row['artifact_name']}
Size: {len(text):,} chars
---
{text}
""")
    
    if not file_contents:
        return {'summary': 'No readable file content available.', 'file_warnings': file_warnings}
    
    all_content = "\n\n".join(file_contents)
    
    prompt = f"""You are analyzing {len(file_contents)} AI coding tool configuration files from Cluster {cluster_id}.

These files include:
- {len(exemplars[exemplars['selection_type'] == 'centroid_nearest'])} files nearest to cluster centroid (most representative)
- {len(exemplars[exemplars['selection_type'] == 'boundary'])} boundary files (edge cases within the cluster)

Analyze these files and answer the following research questions:

1. **Content Theme**: What are these files primarily about? What specific topics, technologies, or domains do they address?

2. **AI Agent Usage**: How might an AI coding agent use the information in these files? What guidance do they provide?

3. **Patterns & Conventions**: What patterns or conventions are being established for AI behavior? What rules, constraints, or preferences are specified?

4. **Variation within Cluster**: Based on the centroid vs boundary files, how much variation exists within this cluster? Are the boundary files still clearly related to the core theme?

Be SPECIFIC and cite examples from the actual file content. Focus on unique characteristics that distinguish this cluster.

FILES:
{all_content}"""

    system_msg = """You are an expert researcher analyzing AI coding tool configurations.
Provide detailed, evidence-based analysis. Cite specific examples from the files.
Format your response with clear headers for each question."""

    if USE_ANTHROPIC:
        response = client.messages.create(
            model=ANTHROPIC_MODEL,
            max_tokens=2000,
            system=system_msg,
            messages=[{"role": "user", "content": prompt}]
        )
        summary = response.content[0].text
    else:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": system_msg},
                {"role": "user", "content": prompt}
            ],
            max_tokens=2000,
            temperature=0.3
        )
        summary = response.choices[0].message.content
    
    return {
        'summary': summary,
        'file_warnings': file_warnings,
        'n_files_analyzed': len(file_contents),
        'total_chars': sum(len(row['text']) for _, row in exemplars.iterrows() if row['text'])
    }


print("✓ LLM analysis function defined")
print(f"  - Uses FULL file content (no truncation)")
print(f"  - Asks research-focused questions")
print(f"  - Reports file size warnings")

In [ ]:
# ============================================================
# ANALYZE ALL CLUSTERS (One by One)
# ============================================================
# Loops through all clusters, showing exemplars and LLM analysis for each

from IPython.display import display, Markdown, HTML

# Configuration
N_CENTROID_FILES = 8  # Files nearest to centroid
N_BOUNDARY_FILES = 2  # Files furthest from centroid

# Get all valid cluster IDs (exclude noise = -1)
cluster_ids = sorted(set(oss_labels) - {-1})
print(f"Analyzing {len(cluster_ids)} clusters: {cluster_ids}")
print("=" * 70)

# Store all results
all_cluster_results = {}
cluster_llm_summaries = {}

for cluster_id in cluster_ids:
    
    # ========== HEADER ==========
    display(Markdown(f"""
---
# 🔹 CLUSTER {cluster_id}
"""))
    
    # ========== CLUSTER STATS ==========
    cluster_size = (oss_results['cluster'] == cluster_id).sum()
    n_repos_in_cluster = oss_results[oss_results['cluster'] == cluster_id]['repo_name'].nunique()
    print(f"Cluster size: {cluster_size:,} files from {n_repos_in_cluster} repositories")
    
    # ========== SELECT EXEMPLARS ==========
    exemplars = select_cluster_exemplars(
        cluster_id, 
        n_centroid=N_CENTROID_FILES, 
        n_boundary=N_BOUNDARY_FILES
    )
    
    if exemplars.empty:
        print("⚠ No exemplars found for this cluster")
        continue
    
    n_centroid = len(exemplars[exemplars['selection_type'] == 'centroid_nearest'])
    n_boundary = len(exemplars[exemplars['selection_type'] == 'boundary'])
    repos_in_exemplars = exemplars['repo_name'].nunique()
    
    print(f"Selected {len(exemplars)} exemplars ({n_centroid} centroid, {n_boundary} boundary) from {repos_in_exemplars} repos")
    
    # ========== DISPLAY EXEMPLAR FILES ==========
    exemplar_list = "\n".join([
        f"{'⭐' if row['selection_type'] == 'centroid_nearest' else '🔷'} **{row['repo_name']}** / `{row['artifact_name']}` ({row['file_size']:,} chars)"
        for _, row in exemplars.iterrows()
    ])
    
    display(Markdown(f"""
### Exemplar Files

{exemplar_list}
"""))
    
    # ========== RUN LLM ANALYSIS ==========
    print(f"Running LLM analysis...")
    analysis = analyze_cluster_exemplars(cluster_id, exemplars)
    
    if analysis['file_warnings']:
        for warning in analysis['file_warnings']:
            print(f"  ⚠ {warning}")
    
    print(f"✓ Analyzed {analysis['n_files_analyzed']} files ({analysis['total_chars']:,} chars)")
    
    # ========== DISPLAY LLM RESULTS ==========
    display(Markdown(f"""
### LLM Analysis

{analysis['summary']}
"""))
    
    # ========== STORE RESULTS ==========
    all_cluster_results[cluster_id] = {
        'cluster_size': cluster_size,
        'n_repos': n_repos_in_cluster,
        'exemplars': exemplars,
        'analysis': analysis
    }
    cluster_llm_summaries[cluster_id] = analysis['summary']

print("\n" + "=" * 70)
print(f"✓ COMPLETE: Analyzed {len(all_cluster_results)} clusters")
print("=" * 70)

In [ ]:
# ============================================================
# NAME CLUSTERS (Using LLM)
# ============================================================
# Generates a short, descriptive name for each cluster based on the analysis

def generate_cluster_name(cluster_id, summary, max_retries=2):
    """
    Ask LLM to generate a short cluster name based on the analysis summary.
    
    Args:
        cluster_id: The cluster ID
        summary: The LLM analysis summary for this cluster
        max_retries: Number of retries on failure
    
    Returns:
        str: A short (2-5 word) descriptive name for the cluster
    """
    prompt = f"""Based on this analysis of Cluster {cluster_id}, generate a SHORT descriptive name (2-5 words) that captures the primary theme or purpose of the files in this cluster.

The name should be:
- Concise (2-5 words)
- Descriptive of the main content/purpose
- Suitable as a category label

Analysis:
{summary[:2000]}  # Truncate if very long

Respond with ONLY the cluster name, nothing else. No quotes, no explanation."""

    for attempt in range(max_retries + 1):
        try:
            if USE_ANTHROPIC:
                response = client.messages.create(
                    model=ANTHROPIC_MODEL,
                    max_tokens=50,
                    system="You are a naming expert. Respond with only the cluster name, 2-5 words.",
                    messages=[{"role": "user", "content": prompt}]
                )
                name = response.content[0].text.strip()
            else:
                response = client.chat.completions.create(
                    model="gpt-4o-mini",
                    messages=[
                        {"role": "system", "content": "You are a naming expert. Respond with only the cluster name, 2-5 words."},
                        {"role": "user", "content": prompt}
                    ],
                    max_tokens=50,
                    temperature=0.3
                )
                name = response.choices[0].message.content.strip()
            
            # Clean up the name (remove quotes if present)
            name = name.strip('"\'')
            return name
            
        except Exception as e:
            if attempt < max_retries:
                print(f"  Retry {attempt + 1} for cluster {cluster_id}...")
                continue
            return f"Cluster {cluster_id}"  # Fallback
    
    return f"Cluster {cluster_id}"


# Generate names for all analyzed clusters
print("=" * 70)
print("GENERATING CLUSTER NAMES")
print("=" * 70)

cluster_names = {}

for cluster_id, summary in cluster_llm_summaries.items():
    print(f"Naming Cluster {cluster_id}...", end=" ")
    name = generate_cluster_name(cluster_id, summary)
    cluster_names[cluster_id] = name
    print(f"→ \"{name}\"")

# Display summary table
print("\n" + "=" * 70)
print("CLUSTER NAMES SUMMARY")
print("=" * 70)

for cluster_id in sorted(cluster_names.keys()):
    size = all_cluster_results[cluster_id]['cluster_size']
    name = cluster_names[cluster_id]
    print(f"  Cluster {cluster_id:2d}: {name:<40} ({size:,} files)")

print("=" * 70)

# Add names to results
oss_results['cluster_name'] = oss_results['cluster'].map(cluster_names)
print(f"\n✓ Cluster names added to oss_results DataFrame")

### 6a. Visualize Clusters with Semantic Names

Now that we have descriptive names for each cluster, let's visualize both the training data and OSS data with proper labels instead of cluster numbers.

In [ ]:
# ============================================================
# VISUALIZE TRAINING & OSS DATA WITH CLUSTER NAMES
# ============================================================
# Shows both datasets with semantic cluster labels instead of numbers

from plotly.subplots import make_subplots

# Load training data for visualization
TRAINING_DATA_DIR = Path('../data')
training_assignments_path = TRAINING_DATA_DIR / 'cluster_assignments.csv'
training_embeddings_path = TRAINING_DATA_DIR / 'training_embeddings.pkl'

# Check if we have training data with 2D coordinates
training_viz_available = False

if training_assignments_path.exists():
    training_df = pd.read_csv(training_assignments_path)
    
    # Check if 2D coordinates exist, otherwise compute them
    if 'umap_x' not in training_df.columns:
        print("Computing 2D coordinates for training data...")
        
        # Load training embeddings
        if training_embeddings_path.exists():
            with open(training_embeddings_path, 'rb') as f:
                train_emb_data = pickle.load(f)
            train_embeddings = train_emb_data['embeddings']
            
            # Transform to UMAP space and then to 2D
            train_umap = umap_reducer.transform(train_embeddings)
            train_coords_2d = reducer_2d.transform(train_umap)
            
            training_df['umap_x'] = train_coords_2d[:, 0]
            training_df['umap_y'] = train_coords_2d[:, 1]
            training_viz_available = True
            print(f"✓ Computed 2D coordinates for {len(training_df):,} training files")
        else:
            print(f"⚠ Training embeddings not found at {training_embeddings_path}")
    else:
        training_viz_available = True
        print(f"✓ Training data loaded: {len(training_df):,} files with 2D coordinates")
else:
    print(f"⚠ Training assignments not found at {training_assignments_path}")

# Create cluster name mapping for labels
def get_cluster_label(cluster_id, cluster_names_dict):
    """Get formatted cluster label with name."""
    if cluster_id == -1:
        return "Noise (unassigned)"
    name = cluster_names_dict.get(cluster_id, f"Cluster {cluster_id}")
    return f"{cluster_id}: {name}"

# Apply cluster names to OSS data
oss_results['cluster_label'] = oss_results['cluster'].apply(
    lambda x: get_cluster_label(x, cluster_names)
)

# Create color mapping - consistent colors for both plots
all_cluster_ids = sorted(set(oss_results['cluster'].unique()) - {-1})
color_sequence = px.colors.qualitative.Set3 + px.colors.qualitative.Pastel1
cluster_color_map = {
    get_cluster_label(cid, cluster_names): color_sequence[i % len(color_sequence)]
    for i, cid in enumerate(all_cluster_ids)
}
cluster_color_map["Noise (unassigned)"] = "lightgray"

# ============================================================
# VISUALIZATION 1: OSS Data with Cluster Names
# ============================================================
print("\n" + "=" * 70)
print("VISUALIZATION: OSS Artifacts by Cluster (with semantic names)")
print("=" * 70)

# Sort order for legend: clusters first (sorted), noise last
label_order = [get_cluster_label(cid, cluster_names) for cid in all_cluster_ids] + ["Noise (unassigned)"]

fig_oss_named = px.scatter(
    oss_results,
    x='umap_x',
    y='umap_y',
    color='cluster_label',
    category_orders={'cluster_label': label_order},
    color_discrete_map=cluster_color_map,
    hover_data=['artifact_name', 'repo_name', 'tool_name'],
    title='OSS Artifacts by Cluster (Semantic Names)',
    width=1100,
    height=750,
)

# Style: noise smaller and transparent
fig_oss_named.for_each_trace(
    lambda t: t.update(marker=dict(size=3, opacity=0.2)) 
    if 'Noise' in t.name 
    else t.update(marker=dict(size=6, opacity=0.8))
)

fig_oss_named.update_layout(
    xaxis_title='UMAP Dimension 1',
    yaxis_title='UMAP Dimension 2',
    legend_title='Cluster',
    legend=dict(
        yanchor="top",
        y=0.99,
        xanchor="left",
        x=1.02,
        font=dict(size=10)
    )
)
fig_oss_named.show()

# ============================================================
# VISUALIZATION 2: Side-by-side Training vs OSS (if training available)
# ============================================================
if training_viz_available:
    print("\n" + "=" * 70)
    print("VISUALIZATION: Training Data vs OSS Data Comparison")
    print("=" * 70)
    
    # Apply cluster labels to training data
    training_df['cluster_label'] = training_df['cluster'].apply(
        lambda x: get_cluster_label(x, cluster_names)
    )
    
    # Create side-by-side figure
    fig_comparison = make_subplots(
        rows=1, cols=2,
        subplot_titles=(
            f'Training Data ({len(training_df):,} files)',
            f'OSS Data ({len(oss_results):,} files, {n_clustered:,} clustered)'
        ),
        horizontal_spacing=0.12
    )
    
    # Left plot: Training data
    for cluster_id in [-1] + all_cluster_ids:
        label = get_cluster_label(cluster_id, cluster_names)
        mask = training_df['cluster'] == cluster_id
        
        if mask.sum() == 0:
            continue
            
        color = cluster_color_map.get(label, 'gray')
        
        fig_comparison.add_trace(
            go.Scatter(
                x=training_df.loc[mask, 'umap_x'],
                y=training_df.loc[mask, 'umap_y'],
                mode='markers',
                marker=dict(
                    size=3 if cluster_id == -1 else 5,
                    color=color,
                    opacity=0.2 if cluster_id == -1 else 0.7
                ),
                name=label,
                legendgroup=label,
                showlegend=True
            ),
            row=1, col=1
        )
    
    # Right plot: OSS data
    for cluster_id in [-1] + all_cluster_ids:
        label = get_cluster_label(cluster_id, cluster_names)
        mask = oss_results['cluster'] == cluster_id
        
        if mask.sum() == 0:
            continue
            
        color = cluster_color_map.get(label, 'gray')
        
        fig_comparison.add_trace(
            go.Scatter(
                x=oss_results.loc[mask, 'umap_x'],
                y=oss_results.loc[mask, 'umap_y'],
                mode='markers',
                marker=dict(
                    size=2 if cluster_id == -1 else 4,
                    color=color,
                    opacity=0.15 if cluster_id == -1 else 0.6
                ),
                name=label,
                legendgroup=label,
                showlegend=False  # Already shown from training
            ),
            row=1, col=2
        )
    
    fig_comparison.update_layout(
        title='Cluster Distribution: Training Data vs OSS Data (with Semantic Names)',
        height=650,
        width=1400,
        legend=dict(
            orientation='v',
            yanchor='top',
            y=1,
            xanchor='left',
            x=1.02,
            font=dict(size=9)
        )
    )
    
    fig_comparison.update_xaxes(title_text='UMAP Dim 1')
    fig_comparison.update_yaxes(title_text='UMAP Dim 2', col=1)
    
    fig_comparison.show()

# ============================================================
# CLUSTER SIZE COMPARISON BAR CHART
# ============================================================
print("\n" + "=" * 70)
print("CLUSTER SIZE COMPARISON")
print("=" * 70)

# Build comparison data
comparison_data = []
for cluster_id in all_cluster_ids:
    label = cluster_names.get(cluster_id, f"Cluster {cluster_id}")
    
    # Training size
    if training_viz_available:
        train_size = (training_df['cluster'] == cluster_id).sum()
    else:
        train_size = cluster_metadata.get(cluster_id, {}).get('size', 0)
    
    # OSS size
    oss_size = (oss_results['cluster'] == cluster_id).sum()
    
    comparison_data.append({
        'Cluster': f"{cluster_id}: {label}",
        'Training': train_size,
        'OSS': oss_size
    })

comparison_df = pd.DataFrame(comparison_data)

# Create grouped bar chart
fig_bars = go.Figure()

fig_bars.add_trace(go.Bar(
    name='Training',
    x=comparison_df['Cluster'],
    y=comparison_df['Training'],
    marker_color='steelblue'
))

fig_bars.add_trace(go.Bar(
    name='OSS (HDBSCAN)',
    x=comparison_df['Cluster'],
    y=comparison_df['OSS'],
    marker_color='coral'
))

fig_bars.update_layout(
    title='Cluster Sizes: Training vs OSS Data',
    xaxis_title='Cluster',
    yaxis_title='Number of Files',
    barmode='group',
    height=500,
    width=1200,
    xaxis_tickangle=-45,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5)
)

fig_bars.show()

# Print summary table
print("\nCluster Size Summary:")
print("-" * 70)
print(f"{'Cluster':<45} {'Training':>10} {'OSS':>10}")
print("-" * 70)
for _, row in comparison_df.iterrows():
    print(f"{row['Cluster']:<45} {row['Training']:>10,} {row['OSS']:>10,}")
print("-" * 70)
print(f"{'TOTAL':<45} {comparison_df['Training'].sum():>10,} {comparison_df['OSS'].sum():>10,}")

In [ ]:
# ============================================================
# SUMMARY: All Clusters Overview
# ============================================================

display(Markdown("""
---
# 📊 Cluster Analysis Summary
"""))

summary_data = []
for cluster_id, result in sorted(all_cluster_results.items()):
    exemplars = result['exemplars']
    summary_data.append({
        'Cluster': cluster_id,
        'Files': result['cluster_size'],
        'Repos': result['n_repos'],
        'Exemplars': len(exemplars),
        'Exemplar Repos': exemplars['repo_name'].nunique(),
        'Total Chars Analyzed': f"{result['analysis']['total_chars']:,}"
    })

summary_df = pd.DataFrame(summary_data)
display(summary_df)

print(f"\nTotal files analyzed across all clusters: {sum(r['cluster_size'] for r in all_cluster_results.values()):,}")
print(f"Results stored in: all_cluster_results, cluster_llm_summaries")

### 6b. LLM Meta-Clustering (Optional)

Use the LLM to consolidate similar clusters into fewer, more meaningful groups.

**Note:** This requires running batch analysis (Section 6a) first to generate summaries for all clusters.

In [ ]:
# ============================================================                          
# META-CLUSTERING: Consolidate clusters into super-categories                           
# ============================================================                          
# Requires cluster_llm_summaries from batch analysis (Section 6a)                       
                                                                                        
if not cluster_llm_summaries or len(cluster_llm_summaries) < 2:                         
    print("⚠ Meta-clustering requires batch analysis (Section 6a) to run first.")       
    print("  Set RUN_BATCH_ANALYSIS = True and re-run Section 6a.")                     
else:                                                                                   
    # Build clusters_text from cluster_llm_summaries WITH cluster names                 
    clusters_text = "\n\n".join([                                                       
        f"Cluster {cid} ({cluster_names.get(cid, 'Unknown')}):\n{summary[:500]}..."     
        if len(summary) > 500                                                           
        else f"Cluster {cid} ({cluster_names.get(cid, 'Unknown')}):\n{summary}"         
        for cid, summary in sorted(cluster_llm_summaries.items())                       
    ])                                                                                  
                                                                                        
    n_clusters = len(cluster_llm_summaries)                                             
                                                                                        
    # Print cluster names for reference                                                 
    print("=" * 70)                                                                     
    print("CLUSTERS TO CONSOLIDATE")                                                    
    print("=" * 70)                                                                     
    for cid in sorted(cluster_llm_summaries.keys()):                                    
        name = cluster_names.get(cid, 'Unknown')                                        
        size = (oss_results['cluster'] == cid).sum()                                    
        print(f"  Cluster {cid:2d}: {name:<40} ({size:,} files)")                       
    print("=" * 70 + "\n")                                                              
                                                                                        
    meta_prompt = f"""You are analyzing {n_clusters} clusters of AI coding tool         
configuration files (like .cursorrules, CLAUDE.md, .aider.conf.yml).                    
                                                                                        
Here are the current clusters with their names and summaries:                           
{clusters_text}                                                                         
                                                                                        
TASK: Consolidate these {n_clusters} clusters into 5-8 MEANINGFUL super-categories.     
                                                                                        
For each super-category:                                                                
1. Give it a SHORT descriptive name (2-4 words)                                         
2. List which cluster IDs belong to it                                                  
3. Write a 1-sentence description of what unifies these clusters                        
                                                                                        
Respond in this exact format:                                                           
SUPER-CATEGORY:                                                                         
CLUSTERS:                                                                               
DESCRIPTION: <1 sentence>                                                               
                                                                                        
Focus on SEMANTIC similarity, not just file types. What are developers ACTUALLY trying  
to accomplish?                                                                          
"""                                                                                     
                                                                                        
    print(f"Asking LLM to consolidate {n_clusters} clusters...")                        
    print(f"Using model: {ANTHROPIC_MODEL}")                                            
                                                                                        
    if USE_ANTHROPIC:                                                                   
        response = client.messages.create(                                              
            model=ANTHROPIC_MODEL,                                                      
            max_tokens=1500,                                                            
            system="You are an expert at categorizing software development patterns. Be concise.",                                                                              
            messages=[{"role": "user", "content": meta_prompt}]                         
        )                                                                               
        meta_clustering_result = response.content[0].text                               
    else:                                                                               
        response = client.chat.completions.create(                                      
            model="gpt-4o-mini",                                                        
            messages=[                                                                  
                {"role": "system", "content": "You are an expert at categorizing software development patterns. Be concise."},                                           
                {"role": "user", "content": meta_prompt}                                
            ],                                                                          
            max_tokens=1500,                                                            
            temperature=0.3                                                             
        )                                                                               
        meta_clustering_result = response.choices[0].message.content                    
                                                                                        
    print("✓ Meta-clustering complete\n")                                               
    print(meta_clustering_result) 

In [ ]:
# Parse meta-clustering result and create mapping                                       
import re                                                                               
                                                                                        
super_categories = {}                                                                   
cluster_to_super = {}                                                                   
                                                                                        
if 'meta_clustering_result' not in dir() or not meta_clustering_result:                 
    print("⚠ No meta-clustering result available.")                                     
    print("  Run the meta-clustering cell above first (requires batch analysis).")      
else:                                                                                   
    # Parse the structured output                                                       
    pattern = r'SUPER-CATEGORY:\s*(.+?)\nCLUSTERS:\s*(.+?)\nDESCRIPTION:\s*(.+?)(?=\n\nSUPER-CATEGORY:|$)'                                                                      
    matches = re.findall(pattern, meta_clustering_result, re.DOTALL)                    
                                                                                        
    for name, clusters_str, description in matches:                                     
        name = name.strip()                                                             
        description = description.strip()                                               
                                                                                        
        # Parse cluster IDs                                                             
        cluster_ids_parsed = [int(c.strip()) for c in re.findall(r'\d+', clusters_str)] 
                                                                                        
        super_categories[name] = {                                                      
            'clusters': cluster_ids_parsed,                                             
            'description': description,                                                 
            'n_files': sum(len(oss_results[oss_results['cluster'] == c]) for c in       
cluster_ids_parsed)                                                                     
        }                                                                               
                                                                                        
        for c in cluster_ids_parsed:                                                    
            cluster_to_super[c] = name                                                  
                                                                                        
    # Add super-category to results                                                     
    oss_results['super_category'] = oss_results['cluster'].map(cluster_to_super)        
                                                                                        
    print(f"Created {len(super_categories)} super-categories:\n")                       
    print("=" * 70)                                                                     
    for super_name, info in super_categories.items():                                   
        print(f"\n📁 {super_name}")                                                     
        print(f"   {info['description']}")                                              
        print(f"   Files: {info['n_files']:,}")                                         
        print(f"   Clusters:")                                                          
        for cid in info['clusters']:                                                    
            cname = cluster_names.get(cid, 'Unknown')                                   
            csize = (oss_results['cluster'] == cid).sum()                               
            print(f"      • {cid}: {cname} ({csize:,} files)")                          
    print("\n" + "=" * 70)   

In [ ]:
# Visualize super-categories
fig = px.scatter(
    oss_results.dropna(subset=['super_category']),
    x='umap_x',
    y='umap_y',
    color='super_category',
    hover_data=['artifact_name', 'repo_name', 'cluster'],
    title='OSS Artifacts by Super-Category (LLM-consolidated)',
    width=1000,
    height=700,
)

fig.update_traces(marker=dict(size=5, opacity=0.7))
fig.show()

# Bar chart of super-category sizes
super_cat_sizes = oss_results['super_category'].value_counts()
fig2 = px.bar(
    x=super_cat_sizes.index,
    y=super_cat_sizes.values,
    title='Files per Super-Category',
    labels={'x': 'Super-Category', 'y': 'Number of Files'}
)
fig2.update_layout(xaxis_tickangle=-45)
fig2.show()

## 7. Export Results

In [ ]:
# ============================================================
# EXPORT RESULTS
# ============================================================

import json

# Save cluster assignments (includes super_category if meta-clustering was run)
output_csv = DATA_DIR / 'oss_cluster_assignments.csv'
oss_results.to_csv(output_csv, index=False)
print(f"✓ Cluster assignments saved to: {output_csv}")

# Save LLM summaries as readable JSON (from batch or single analysis)
if cluster_llm_summaries:
    summaries_json_path = DATA_DIR / 'oss_cluster_summaries.json'
    summaries_for_json = {int(k): v for k, v in cluster_llm_summaries.items()}
    with open(summaries_json_path, 'w') as f:
        json.dump(summaries_for_json, f, indent=2)
    print(f"✓ LLM summaries saved to: {summaries_json_path}")
    print(f"    ({len(cluster_llm_summaries)} cluster summaries)")
else:
    print("⚠ No LLM summaries to save (run batch analysis in Section 6a)")

# Save super-categories (if meta-clustering was run)
if super_categories:
    super_cat_path = DATA_DIR / 'oss_super_categories.json'
    super_cat_for_json = {
        name: {
            'clusters': info['clusters'],
            'description': info['description'],
            'n_files': info['n_files']
        }
        for name, info in super_categories.items()
    }
    with open(super_cat_path, 'w') as f:
        json.dump(super_cat_for_json, f, indent=2)
    print(f"✓ Super-categories saved to: {super_cat_path}")
    print(f"    ({len(super_categories)} categories)")
else:
    print("⚠ No super-categories to save (run meta-clustering in Section 6b)")

# Save detailed cluster analyses (if batch analysis was run)
if 'cluster_analyses' in dir() and cluster_analyses:
    # Save exemplar file lists per cluster
    exemplar_data = {}
    for cluster_id, result in cluster_analyses.items():
        exemplars = result.get('exemplars', pd.DataFrame())
        if not exemplars.empty:
            exemplar_data[int(cluster_id)] = {
                'n_exemplars': len(exemplars),
                'n_repos': exemplars['repo_name'].nunique(),
                'total_chars': result['analysis'].get('total_chars', 0),
                'files': exemplars[['file_id', 'artifact_name', 'repo_name', 'selection_type']].to_dict('records')
            }
    
    exemplars_path = DATA_DIR / 'oss_cluster_exemplars.json'
    with open(exemplars_path, 'w') as f:
        json.dump(exemplar_data, f, indent=2)
    print(f"✓ Cluster exemplars saved to: {exemplars_path}")

print(f"\n{'='*70}")
print("EXPORT SUMMARY")
print("=" * 70)
print(f"  Total OSS files: {len(oss_results):,}")
print(f"  Clusters analyzed: {len(cluster_llm_summaries) if cluster_llm_summaries else 0}")
print(f"  Super-categories: {len(super_categories) if super_categories else 0}")
if -1 in set(oss_labels):
    print(f"  Noise/unassigned: {(oss_labels == -1).sum():,}")

## 8. Cluster Comparison: Training vs OSS

Compare the semantic composition of clusters between the training data and OSS data.

In [ ]:
# Compare training cluster metadata with OSS assignments
print("Cluster Comparison: Training Data vs OSS")
print("=" * 70)

# Build basic OSS cluster stats for comparison
oss_cluster_stats = {}
for c in sorted(set(oss_labels) - {-1}):
    cluster_data = oss_results[oss_results['cluster'] == c]
    oss_cluster_stats[c] = {
        'size': len(cluster_data),
        'top_tools': cluster_data['tool_name'].value_counts().head(5).to_dict(),
        'top_artifacts': cluster_data['artifact_name'].value_counts().head(5).to_dict(),
    }

for c in sorted(cluster_metadata.keys()):
    train_meta = cluster_metadata[c]
    oss_meta = oss_cluster_stats.get(c, {})
    
    print(f"\n{'='*70}")
    print(f"CLUSTER {c}")
    print(f"{'='*70}")
    print(f"\n  Training: {train_meta['size']:,} files")
    print(f"  OSS: {oss_meta.get('size', 0):,} files")
    
    print(f"\n  Training top tools: {list(train_meta['top_tools'].keys())[:3]}")
    print(f"  OSS top tools: {list(oss_meta.get('top_tools', {}).keys())[:3]}")
    
    print(f"\n  Training top artifacts: {list(train_meta['top_artifacts'].keys())[:3]}")
    print(f"  OSS top artifacts: {list(oss_meta.get('top_artifacts', {}).keys())[:3]}")

## 9. Cluster Combinations per Repository

**Research Question:** Do repositories use a single cluster (homogeneous) or multiple clusters (blended/layered)?

This analysis tests whether:
- Repositories intentionally layer different types of content (AI instructions + context + docs)
- Or if cluster diversity indicates lack of standardization

**Hypothesis:** If blending is intentional, we should see consistent cluster combination patterns across repositories.

In [ ]:
# Load training data cluster assignments
TRAINING_DATA_DIR = Path('../data')
training_assignments_path = TRAINING_DATA_DIR / 'cluster_assignments.csv'

if training_assignments_path.exists():
    training_df = pd.read_csv(training_assignments_path)
    print(f"Loaded training cluster assignments: {len(training_df):,} files")
    print(f"Repositories: {training_df['repo_name'].nunique():,}")
    print(f"Clusters: {sorted(training_df['cluster'].unique())}")
else:
    print(f"Training assignments not found at: {training_assignments_path}")
    print("Run notebook 4 (embedding_natural_clusters) first.")

In [ ]:
# Analyze cluster combinations per repository
from itertools import combinations

def get_cluster_signature(clusters):
    """Create a canonical signature for a set of clusters."""
    unique = sorted(set(clusters))
    return tuple(unique)

# Group by repository and get cluster sets
repo_clusters = training_df.groupby('repo_name')['cluster'].apply(list).to_dict()

# Calculate statistics per repo
repo_stats = []
for repo, clusters in repo_clusters.items():
    unique_clusters = set(clusters)
    # Exclude noise (-1) from cluster count
    non_noise_clusters = unique_clusters - {-1}
    
    repo_stats.append({
        'repo_name': repo,
        'n_files': len(clusters),
        'n_clusters': len(non_noise_clusters),
        'clusters': get_cluster_signature(non_noise_clusters),
        'has_noise': -1 in unique_clusters,
        'noise_pct': clusters.count(-1) / len(clusters) * 100 if clusters else 0
    })

repo_stats_df = pd.DataFrame(repo_stats)

print("=" * 70)
print("REPOSITORY CLUSTER DIVERSITY")
print("=" * 70)
print(f"\nTotal repositories: {len(repo_stats_df):,}")
print(f"\nCluster count distribution:")
print(repo_stats_df['n_clusters'].value_counts().sort_index())

In [ ]:
# Visualize cluster count distribution
fig_dist = px.histogram(
    repo_stats_df,
    x='n_clusters',
    title='Distribution of Cluster Count per Repository',
    labels={'n_clusters': 'Number of Distinct Clusters', 'count': 'Number of Repositories'},
    nbins=max(repo_stats_df['n_clusters']) + 1
)
fig_dist.update_layout(bargap=0.1)
fig_dist.show()

# Calculate percentages
n_single = (repo_stats_df['n_clusters'] == 1).sum()
n_multi = (repo_stats_df['n_clusters'] > 1).sum()
total = len(repo_stats_df)

print(f"\n{'='*70}")
print(f"HOMOGENEOUS vs BLENDED REPOSITORIES")
print(f"{'='*70}")
print(f"  Single-cluster (homogeneous): {n_single:,} ({n_single/total*100:.1f}%)")
print(f"  Multi-cluster (blended): {n_multi:,} ({n_multi/total*100:.1f}%)")

In [ ]:
# Analyze most common cluster combinations
from collections import Counter

# Get combination frequencies (only for repos with 2+ clusters)
multi_cluster_repos = repo_stats_df[repo_stats_df['n_clusters'] >= 2]
combination_counts = Counter(multi_cluster_repos['clusters'].tolist())

print("=" * 70)
print("MOST COMMON CLUSTER COMBINATIONS (repos with 2+ clusters)")
print("=" * 70)
print(f"\nTotal multi-cluster repos: {len(multi_cluster_repos):,}")
print(f"Unique combinations: {len(combination_counts):,}")

print(f"\nTop 20 combinations:")
for combo, count in combination_counts.most_common(20):
    pct = count / len(multi_cluster_repos) * 100
    combo_str = ', '.join(map(str, combo))
    print(f"  ({combo_str}): {count} repos ({pct:.1f}%)")

In [ ]:
# Analyze cluster co-occurrence matrix
# Which clusters tend to appear together?

all_clusters = sorted(set(training_df['cluster'].unique()) - {-1})
n_clusters_total = len(all_clusters)

# Build co-occurrence matrix
cooccurrence = np.zeros((n_clusters_total, n_clusters_total), dtype=int)

for clusters in repo_clusters.values():
    unique = list(set(clusters) - {-1})
    for i, c1 in enumerate(unique):
        for c2 in unique[i:]:
            idx1 = all_clusters.index(c1)
            idx2 = all_clusters.index(c2)
            cooccurrence[idx1, idx2] += 1
            if idx1 != idx2:
                cooccurrence[idx2, idx1] += 1

# Create heatmap
fig_cooc = go.Figure(data=go.Heatmap(
    z=cooccurrence,
    x=[f'C{c}' for c in all_clusters],
    y=[f'C{c}' for c in all_clusters],
    colorscale='Blues',
    text=cooccurrence,
    texttemplate='%{text}',
    textfont={"size": 10},
))

fig_cooc.update_layout(
    title='Cluster Co-occurrence Matrix (repos containing both clusters)',
    xaxis_title='Cluster',
    yaxis_title='Cluster',
    width=800,
    height=700
)
fig_cooc.show()

In [ ]:
# Analyze blending patterns by cluster
# For each cluster, what other clusters does it commonly appear with?

print("=" * 70)
print("CLUSTER BLENDING PATTERNS")
print("=" * 70)
print("\nFor each cluster: what % of repos also contain other clusters?\n")

for cluster_id in all_clusters:
    # Get repos containing this cluster
    repos_with_cluster = [
        repo for repo, clusters in repo_clusters.items() 
        if cluster_id in clusters
    ]
    n_repos = len(repos_with_cluster)
    
    if n_repos == 0:
        continue
    
    # Count co-occurring clusters
    co_cluster_counts = Counter()
    for repo in repos_with_cluster:
        other_clusters = set(repo_clusters[repo]) - {cluster_id, -1}
        for oc in other_clusters:
            co_cluster_counts[oc] += 1
    
    # Calculate isolation rate (repos where this is the ONLY cluster)
    isolated_repos = sum(1 for repo in repos_with_cluster 
                         if set(repo_clusters[repo]) - {-1} == {cluster_id})
    isolation_rate = isolated_repos / n_repos * 100
    
    print(f"Cluster {cluster_id} ({n_repos} repos):")
    print(f"  Isolated (only cluster): {isolation_rate:.1f}%")
    
    if co_cluster_counts:
        top_co = co_cluster_counts.most_common(3)
        co_str = ', '.join([f"C{c}({cnt}/{n_repos})" for c, cnt in top_co])
        print(f"  Most common companions: {co_str}")
    print()

### 9.1 Interpretation of Cluster Combinations

**Key Questions Answered:**

1. **Are repositories homogeneous or blended?**
   - Look at the histogram: if most repos have 1 cluster → homogeneous practices
   - If most have 2+ clusters → intentional layering

2. **Are there consistent blending patterns?**
   - Look at top combinations: if a few patterns dominate → intentional strategies
   - If highly diverse → experimental/ad-hoc practices

3. **Which clusters are "standalone" vs "companion"?**
   - High isolation rate → cluster represents complete solution
   - Low isolation rate → cluster is typically used with others

In [ ]:
# Summary statistics
print("=" * 70)
print("SUMMARY: CLUSTER COMBINATION ANALYSIS")
print("=" * 70)

# Key metrics
n_total = len(repo_stats_df)
n_single = (repo_stats_df['n_clusters'] == 1).sum()
n_two = (repo_stats_df['n_clusters'] == 2).sum()
n_three_plus = (repo_stats_df['n_clusters'] >= 3).sum()

avg_clusters = repo_stats_df['n_clusters'].mean()
median_clusters = repo_stats_df['n_clusters'].median()

print(f"\nRepository distribution:")
print(f"  1 cluster (homogeneous): {n_single:,} ({n_single/n_total*100:.1f}%)")
print(f"  2 clusters: {n_two:,} ({n_two/n_total*100:.1f}%)")
print(f"  3+ clusters: {n_three_plus:,} ({n_three_plus/n_total*100:.1f}%)")

print(f"\nCentral tendency:")
print(f"  Mean clusters per repo: {avg_clusters:.2f}")
print(f"  Median clusters per repo: {median_clusters:.1f}")

# Unique combinations
n_unique_combos = len(set(repo_stats_df['clusters']))
print(f"\nDiversity:")
print(f"  Unique cluster combinations: {n_unique_combos:,}")
print(f"  Combination diversity ratio: {n_unique_combos/n_total:.3f}")

# Conclusion
print(f"\n{'='*70}")
if n_multi / n_total > 0.5:
    print("CONCLUSION: Majority of repos use BLENDED approach (multi-cluster)")
    print("This suggests intentional layering of different content types.")
else:
    print("CONCLUSION: Majority of repos are HOMOGENEOUS (single-cluster)")
    print("This suggests focused, single-purpose AI configuration files.")
print("=" * 70)